# 1. Extended thinking

讓 Claude 在回覆前花時間對 task 進行思考，適合複雜的任務使用，需要多消耗時間以及 token

> [使用限制](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking#feature-compatibility)

## How Extended Thinking Works

開啟 extended thinking 功能後，Claude 回覆的內容會包含兩部分:
1. `Text Block` -> Final answer
2. `Thinking Block` -> Reasoning process

![](./figure/05/thinking_01.jpg)

有以下優點以及缺點:
- 優點:
    - Better reasoning capabilities for complex tasks
    - Increased accuracy on difficult problems
    - Transparency into Claude's thought process
- 缺點:
    - Higher costs (you pay for thinking tokens)
    - Increased latency (thinking takes time)
    - More complex response handling in your code

## When to Use Extended Thinking

當用盡一般的 prompt 技巧以及方法後，產生的內容若還是不如你的預期(或沒有達到你要的標準)，此時就可以使用

> 官方原話:
> Run your prompts without thinking first (先不用思考功能，不是叫你不要思考) , and if the accuracy isn't meeting your requirements after you've already optimized your prompt, then consider enabling extended thinking. It's a tool for when standard prompting isn't quite getting you there.

## Response Structure and Security

Claude 會在回覆 thinking 訊息內容中加入一個 signature，來確保後續你在回傳 thinking 紀錄給 Claude 時，內容沒有竄改。
這可以防止 developer 汙染 thinking 內容，避免 Claude 被操控或回覆不安全的內容

![](./figure/05/thinking_02.jpg)

## Redacted Thinking

有時候使用 thinking 時會收到 unreadable 的 Thinking Block，這表示 thinking 過程被 Claude 內部安全系統 flag，並將 原先的 thinking process plaintext 進行加密。後續要當成 thinking 紀錄使用時將這些加密後的內容傳回去即可。

![](./figure/05/thinking_03.png)

## Implementation

詳見 [./source/001_thinking_complete.ipynb](./source/001_thinking_complete.ipynb)

# 2. Image support

Claude 擁有 vision process 能力，可以使用他來處理圖片問題，像是:
- describe what's in an image, 
- compare multiple images, 
- count objects, 
- perform complex visual analysis tasks

## Image Handling Basics

處理圖片時有以下幾個限制需要特別注意:
1. Up to 100 images across all messages in a single request
1. Max size of 5MB per image
1. When sending one image: max height/width of 8000px
1. When sending `multiple` images: max height/width of 2000px
1. Images can be included as `base64` encoding or a `URL` to the image
1. Each image counts as tokens based on its dimensions: `tokens = (width px × height px) / 750`

![](./figure/05/image_01.jpg)

要把圖片傳給 Claude 處理時，需要加入 `Image Block`

架構會像這樣:
```python
with open("image.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "What do you see in this image?"
    }
])
```

Claude 處理後會回傳 Text Block

![](./figure/05/image_02.jpg)

## Prompting Techniques

在處理圖片時，需要好的 prompt engineering 技術。過於簡單的文字描述往往會收到不好的結果

![](./figure/05/image_03.jpg)

可以透過以下方法來大幅增加 Claude 回覆準確度:
- Providing **detailed guidelines** and **analysis steps**
- Using one-shot or multi-shot **examples**
- Breaking down complex tasks into **smaller steps**

### 1. Step-by-Step Analysis

提供 claude 一步步分析的方法，例如:

```python
"""
Analyze this image of marbles and determine the exact count using this methodology:
1. Begin by identifying each unique marble one at a time. Assign each a number as you identify it.
2. Verify your result by counting with a different method. Start from the bottom-left corner and work row by row, from left to right.

What is the exact, verified number of marbles in this image?
"""
```

![](./figure/05/image_04.jpg)

### 2. One-Shot Examples

![](./figure/05/image_05.jpg)

## Real-World Example: Fire Risk Assessment

設計一個 *使用衛星圖片來分析家屋因森林大火而發生火災風險* 的系統，這個系統需要辨別以下事情:
- Dense, close-packed trees near the residence
- Difficult access routes for emergency services
- Branches overhanging the residence

![](./figure/05/image_06.jpg)

相較於簡單的 *提供火災風險分數* 這樣籠統的 prompt，撰寫擁有架構、將分析方法拆解成較小步驟的 prompt 更為合適，例如:
```python
"""
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75%+)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the home
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple vulnerability points
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numerical rating.
"""
```

剩下的程式碼可以看: [./source/002_images.ipynb](./source/002_images.ipynb)